# Chapter 2: Hypothesis Testing — Is This Difference Real, or Just Noise?

### Why This Matters

Descriptive statistics summarize what you observed. **Hypothesis testing** answers a sharper business question: "Sales in Region A were ₹5,000 higher than Region B this month — is that a real, reliable difference, or could it easily be random noise from a small sample?" This distinction is precisely what separates a rigorous analyst from someone who reacts to every fluctuation in a dashboard as if it were meaningful.

### 2.1 The Core Logic

Every hypothesis test starts from the same skeptical default position, called the **null hypothesis (H₀)**: "there is no real difference/effect; what we observed is just random variation." The test then computes a **p-value** — roughly, "if the null hypothesis were actually true, how likely is it we'd see data at least this extreme just by chance?" A small p-value (conventionally, below 0.05) means "this would be quite unlikely under pure chance," giving you grounds to reject the null hypothesis and conclude the effect is likely real.

⚠️ **A p-value is not "the probability the result is due to chance," despite that being the single most common misinterpretation** — including, frequently, in business settings by people who should know better. It is the probability of observing data this extreme (or more) *if the null hypothesis were true*. This is a subtle but real distinction; when in doubt, phrase your conclusion around "this is unlikely to be pure noise" rather than making a claim about the probability of the hypothesis itself being true.

### 2.2 Comparing Two Groups: the t-test

In [ ]:
from scipy import stats

region_a_sales = np.array([102, 98, 110, 105, 99, 108, 101, 95, 112, 104])
region_b_sales = np.array([95, 92, 100, 97, 90, 99, 94, 88, 103, 96])

t_stat, p_value = stats.ttest_ind(region_a_sales, region_b_sales)
print(f"t-statistic: {t_stat:.3f}  |  p-value: {p_value:.4f}")

if p_value < 0.05:
    print("Statistically significant difference between the two regions' sales.")
else:
    print("No statistically significant difference detected — the gap could be random noise.")

An **independent samples t-test** (`ttest_ind`) is the standard tool for exactly this "are these two group averages meaningfully different" question — comparing two marketing campaigns, two store layouts, two product versions in an A/B test, or (as here) two regions. The `0.05` threshold (a 5% chance of falsely concluding there's a difference when there isn't one) is a convention, not a law of nature — some fields and high-stakes decisions use a stricter 0.01, and it's worth stating your threshold explicitly rather than treating 0.05 as sacred.

### 2.3 Comparing Categorical Proportions: the Chi-Square Test

In [ ]:
# Business question: does the proportion of customers who churn differ by subscription plan?
observed = np.array([
    [50, 200],    # Plan A: [churned, retained]
    [30, 220]     # Plan B: [churned, retained]
])

chi2, p_value, dof, expected = stats.chi2_contingency(observed)
print(f"Chi-square: {chi2:.3f}  |  p-value: {p_value:.4f}")

The **chi-square test of independence** answers "are these two categorical variables related, or independent?" — here, "does churn rate depend on which plan a customer is on?" It compares your actual observed counts against what you'd expect if the two variables were completely unrelated (`expected`), and a small p-value indicates the observed pattern is unlikely to arise from pure chance if plan and churn were truly independent — i.e., the plan probably *does* matter for churn.

### 2.4 A Practical Warning About Multiple Testing

⚠️ If you run 20 hypothesis tests on 20 different metrics at the standard 0.05 threshold, you should *expect* roughly one of them to show "statistical significance" purely by chance, even if nothing real is going on anywhere. This is a genuine, frequently-abused trap in business analytics — "we tested 15 marketing variants and Variant 7 was significant!" is exactly the scenario where that "significant" result deserves real scrutiny (was this the single hypothesis you set out to test, or one of many you fished through afterward?) before you act on it. When you must run many comparisons, look into corrections like the Bonferroni correction, or better, pre-register the specific hypothesis you're testing before looking at the data.

### Cheat Sheet — Hypothesis Testing

| Question | Test | Function |
|---|---|---|
| Are two group means different? | Independent t-test | `stats.ttest_ind(group1, group2)` |
| Are two paired measurements different? (before/after) | Paired t-test | `stats.ttest_rel(before, after)` |
| Is a categorical outcome related to a categorical group? | Chi-square | `stats.chi2_contingency(table)` |
| Are more than two group means different? | ANOVA | `stats.f_oneway(g1, g2, g3)` |
| Interpreting the result | p < 0.05 → likely a real effect; p ≥ 0.05 → can't rule out chance | — |

---